# Sesión 03 - Variables aleatorias

Objetivo: transformar resultados aleatorios en variables discretas y continuas, trabajar con PMF, PDF aproximada y CDF, y conectar la likelihood multinomial con el score usado por MultinomialNB.


In [ ]:
from itertools import product
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(7)
pd.set_option("display.precision", 4)


## 0. Seguimiento: de la likelihood multinomial al score logarítmico

En MultinomialNB observamos un vector de conteos $x=(x_1,\ldots,x_d)$. Si $n=\sum_j x_j$, la likelihood multinomial para una clase $c$ es:

$$
P(x\mid Y=c)=\frac{n!}{\prod_{j=1}^{d}x_j!}\prod_{j=1}^{d}\hat{\theta}_{cj}^{x_j}
$$

Queremos llegar al score:

$$
s_c=\log \hat{\pi}_c+\sum_{j=1}^{d}x_j\log \hat{\theta}_{cj}
$$

El paso no es magia algebraica; son cuatro decisiones: usar Bayes, eliminar términos constantes en la comparación entre clases, tomar logaritmos y convertir productos en sumas.


### Derivación paso a paso

1. **Regla de decisión Bayesiana**

$$
\hat{c}=\arg\max_c P(Y=c\mid x)
$$

2. **Sustituimos Bayes**

$$
P(Y=c\mid x)=\frac{P(x\mid Y=c)P(Y=c)}{P(x)}
$$

Como $P(x)$ es el mismo para todas las clases, no cambia el $\arg\max$:

$$
\hat{c}=\arg\max_c P(x\mid Y=c)P(Y=c)
$$

3. **Separamos qué se estima y qué se reemplaza**

El prior de clase se estima con la proporción de registros de la clase $c$:

$$
P(Y=c)\approx \hat{\pi}_c
$$

La likelihood se estima con el modelo multinomial suavizado:

$$
P(x\mid Y=c)\approx \frac{n!}{\prod_{j=1}^{d}x_j!}\prod_{j=1}^{d}\hat{\theta}_{cj}^{x_j}
$$

El prior $\hat{\pi}_c$ no sale de la likelihood; viene de la frecuencia de la clase. La likelihood explica qué tan compatible es el vector de conteos $x$ con esa clase.

4. **Reemplazamos línea por línea**

Primero sustituimos el prior:

$$
\hat{c}=\arg\max_c P(x\mid Y=c)\hat{\pi}_c
$$

Luego sustituimos la likelihood multinomial:

$$
\hat{c}=\arg\max_c \left[\left(\frac{n!}{\prod_{j=1}^{d}x_j!}\prod_{j=1}^{d}\hat{\theta}_{cj}^{x_j}\right)\hat{\pi}_c\right]
$$

Finalmente reordenamos factores por conmutatividad de la multiplicación:

$$
\hat{c}=\arg\max_c \left[\hat{\pi}_c\frac{n!}{\prod_{j=1}^{d}x_j!}\prod_{j=1}^{d}\hat{\theta}_{cj}^{x_j}\right]
$$

5. **Eliminamos el coeficiente multinomial para clasificar**

Para un $x$ fijo,

$$
K(x)=\frac{n!}{\prod_{j=1}^{d}x_j!}
$$

no depende de $c$. Entonces:

$$
\arg\max_c \left[\hat{\pi}_cK(x)\prod_{j=1}^{d}\hat{\theta}_{cj}^{x_j}\right]
=
\arg\max_c \left[\hat{\pi}_c\prod_{j=1}^{d}\hat{\theta}_{cj}^{x_j}\right]
$$

6. **Tomamos logaritmo porque conserva el orden**

$$
\arg\max_c a_c=\arg\max_c \log(a_c),\quad a_c>0
$$

Por tanto:

$$
\arg\max_c \left[\hat{\pi}_c\prod_{j=1}^{d}\hat{\theta}_{cj}^{x_j}\right]
=
\arg\max_c \left[\log\hat{\pi}_c+\log\prod_{j=1}^{d}\hat{\theta}_{cj}^{x_j}\right]
$$

7. **Convertimos producto en suma**

$$
\log\prod_{j=1}^{d}\hat{\theta}_{cj}^{x_j}=\sum_{j=1}^{d}\log\left(\hat{\theta}_{cj}^{x_j}\right)=\sum_{j=1}^{d}x_j\log\hat{\theta}_{cj}
$$

Así obtenemos:

$$
s_c=\log \hat{\pi}_c+\sum_{j=1}^{d}x_j\log \hat{\theta}_{cj}
$$

El coeficiente multinomial se puede omitir para decidir la clase, pero se mantiene si queremos calcular la probabilidad exacta $P(x\mid Y=c)$.


In [ ]:
import math

features = ["cafe", "frio", "fila", "rapido"]
classes = np.array(["queja", "no_queja"])

# Conteos agregados N_cj por clase y feature.
N_cj = np.array([
    [4, 5, 3, 0],  # queja
    [3, 0, 1, 5],  # no_queja
])

class_count = np.array([3, 3])
alpha = 1.0
d = N_cj.shape[1]

# Paso 1: prior de clase.
pi_hat = class_count / class_count.sum()

# Paso 2: theta suavizado.
theta_hat = (N_cj + alpha) / (N_cj.sum(axis=1, keepdims=True) + alpha * d)

# Documento nuevo: una vez cafe, frio y fila; cero veces rapido.
x_new = np.array([1, 1, 1, 0])
n = x_new.sum()

# Paso 3: likelihood multinomial completa.
multinomial_coef = math.factorial(n) / np.prod([math.factorial(int(xj)) for xj in x_new])
likelihood_completa = multinomial_coef * np.prod(theta_hat ** x_new, axis=1)
posterior_no_normalizado = pi_hat * likelihood_completa

# Paso 4: score con el coeficiente y score sin el coeficiente.
score_con_coef = np.log(pi_hat) + np.log(multinomial_coef) + x_new @ np.log(theta_hat).T
score_sin_coef = np.log(pi_hat) + x_new @ np.log(theta_hat).T

# Paso 5: normalización del posterior a partir del score sin coeficiente.
posterior = np.exp(score_sin_coef - score_sin_coef.max())
posterior = posterior / posterior.sum()

pd.DataFrame({
    "clase": classes,
    "pi_hat": pi_hat,
    "P(x|clase) completa": likelihood_completa,
    "pi_hat * P(x|clase)": posterior_no_normalizado,
    "score con coef": score_con_coef,
    "score sin coef": score_sin_coef,
    "posterior normalizado": posterior,
})


## 1. Una variable aleatoria discreta sobre dos dados

Definimos $X$ como la suma de dos dados.


### Lectura matemática

- **Distribución asumida:** distribución inducida por una función $X:\Omega	o\mathbb{R}$ sobre un espacio uniforme.
- **Parámetro estimado:** ninguno si enumeramos $\Omega$; la PMF es exacta.
- **Supuesto que puede fallar:** equiprobabilidad del espacio muestral.
- **Diagnóstico:** verificar que la PMF suma 1 y que la CDF es no decreciente.


In [ ]:
omega = np.array(list(product(range(1, 7), repeat=2)))
X = omega.sum(axis=1)

valores, conteos = np.unique(X, return_counts=True)
pmf = pd.DataFrame({"x": valores, "P(X=x)": conteos / conteos.sum()})
pmf["F(X<=x)"] = pmf["P(X=x)"].cumsum()
pmf


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(pmf["x"], pmf["P(X=x)"])
axes[0].set_title("PMF de X = suma de dos dados")
axes[0].set_xlabel("x")
axes[0].set_ylabel("probabilidad")

axes[1].step(pmf["x"], pmf["F(X<=x)"], where="post")
axes[1].set_title("CDF discreta")
axes[1].set_xlabel("x")
axes[1].set_ylabel("F(x)")
axes[1].set_ylim(0, 1.05)
plt.show()


## 2. Transformaciones de variables aleatorias

Del mismo experimento podemos derivar otra variable: $Y = 1\{X \ge 10\}$.


In [ ]:
Y = (X >= 10).astype(int)
pd.Series(Y).value_counts(normalize=True).sort_index().rename("P(Y=y)").to_frame()


## 3. Variable continua: tiempo de espera

Simulamos tiempos de espera exponenciales. En variables continuas no se pregunta por un punto exacto, sino por intervalos.


### Lectura matemática

- **Distribución asumida:** Exponencial con tasa $\lambda$; modela tiempos entre eventos.
- **Parámetro estimado/usado:** $\lambda=1/E[T]$.
- **Supuesto que puede fallar:** tasa constante y falta de memoria.
- **Diagnóstico:** histograma, CDF empírica y comparación de probabilidades por intervalos.


In [ ]:
tasa = 1 / 8  # espera media de 8 minutos
espera = rng.exponential(scale=1 / tasa, size=20_000)

p_entre_5_y_10_emp = np.mean((espera >= 5) & (espera <= 10))
p_entre_5_y_10_teo = stats.expon(scale=1 / tasa).cdf(10) - stats.expon(scale=1 / tasa).cdf(5)

print(f"P(5 <= T <= 10) empírica: {p_entre_5_y_10_emp:.4f}")
print(f"P(5 <= T <= 10) teórica:  {p_entre_5_y_10_teo:.4f}")
print(f"P(T = 8) en modelo continuo: 0")


In [ ]:
xs = np.linspace(0, 40, 300)
pdf = stats.expon(scale=1 / tasa).pdf(xs)
cdf = stats.expon(scale=1 / tasa).cdf(xs)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(espera, bins=40, density=True, alpha=0.45, label="simulación")
axes[0].plot(xs, pdf, color="crimson", label="PDF teórica")
axes[0].set_title("PDF de tiempo de espera")
axes[0].legend()

axes[1].plot(xs, cdf, color="darkgreen")
axes[1].set_title("CDF de tiempo de espera")
axes[1].set_ylim(0, 1.02)
plt.show()


## 4. Estandarización

La transformación $Z=(X-\mu)/\sigma$ permite comparar variables en escalas distintas.


In [ ]:
z_espera = (espera - espera.mean()) / espera.std(ddof=0)
resumen = pd.Series(z_espera).describe(percentiles=[0.05, 0.5, 0.95])
resumen


## 5. Lectura de variables desde fuentes de datos

Si `data_sources/sales_data.csv` está disponible, este bloque identifica variables discretas, continuas, binarias y categóricas del dataset de demanda usado en el material original.


In [ ]:
from pathlib import Path

def display(obj):
    try:
        from IPython.display import display as ipy_display
        ipy_display(obj)
    except Exception:
        if hasattr(obj, "to_string"):
            print(obj.to_string())
        else:
            print(obj)

def encontrar_data_dir():
    for candidato in [Path("data_sources"), Path("../data_sources")]:
        if candidato.exists():
            return candidato
    return None

DATA_DIR = encontrar_data_dir()
if DATA_DIR is None or not (DATA_DIR / "sales_data.csv").exists():
    print("No se encontró data_sources/sales_data.csv. Se mantiene la práctica con datos sintéticos.")
else:
    ventas_df = pd.read_csv(DATA_DIR / "sales_data.csv", nrows=5_000, parse_dates=["Date"])
    catalogo = pd.DataFrame(
        {
            "columna": ventas_df.columns,
            "dtype": [str(ventas_df[c].dtype) for c in ventas_df.columns],
            "n_unicos": [ventas_df[c].nunique() for c in ventas_df.columns],
            "ejemplo": [ventas_df[c].dropna().iloc[0] for c in ventas_df.columns],
        }
    )
    catalogo["lectura_probabilistica"] = np.select(
        [
            catalogo["n_unicos"].eq(2),
            catalogo["dtype"].str.contains("int|float") & catalogo["n_unicos"].le(30),
            catalogo["dtype"].str.contains("int|float"),
            catalogo["dtype"].str.contains("datetime"),
        ],
        ["binaria", "discreta/conteo", "continua o discreta amplia", "índice temporal"],
        default="categórica",
    )
    display(catalogo)


## Práctica

Define una nueva variable aleatoria sobre el lanzamiento de dos dados: máximo, mínimo, diferencia absoluta o indicador de dobles. Construye su PMF y CDF.
